In [2]:
print("Notebook is working")


Notebook is working


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [4]:
df = pd.read_csv("E0.csv")
df = df[['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]
df.head()

,HomeTeam,AwayTeam,FTHG,FTAG
0,Liverpool,Bournemouth,4,2
1,Aston Villa,Newcastle,0,0
2,Brighton,Fulham,1,1
3,Sunderland,West Ham,3,0
4,Tottenham,Burnley,3,0


In [5]:
avg_home_goals = df['FTHG'].mean()
avg_away_goals = df['FTAG'].mean()

home_goals_scored = df.groupby('HomeTeam')['FTHG'].mean()
home_goals_conceded = df.groupby('HomeTeam')['FTAG'].mean()

away_goals_scored = df.groupby('AwayTeam')['FTAG'].mean()
away_goals_conceded = df.groupby('AwayTeam')['FTHG'].mean()

home_attack = home_goals_scored / avg_home_goals
home_defence = home_goals_conceded / avg_away_goals

away_attack = away_goals_scored / avg_away_goals
away_defence = away_goals_conceded / avg_home_goals

print("Average home goals:", round(avg_home_goals, 2))
print("Average away goals:", round(avg_away_goals, 2))


Average home goals: 1.5
Average away goals: 1.23


In [6]:
def predict_match(home_team, away_team, max_goals=8):
    home_exp_goals = avg_home_goals * home_attack[home_team] * away_defence[away_team]
    away_exp_goals = avg_away_goals * away_attack[away_team] * home_defence[home_team]

    home_win = 0
    draw = 0
    away_win = 0

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            prob = poisson.pmf(home_goals, home_exp_goals) * poisson.pmf(away_goals, away_exp_goals)

            if home_goals > away_goals:
                home_win += prob
            elif home_goals == away_goals:
                draw += prob
            else:
                away_win += prob

    if home_win > draw and home_win > away_win:
        prediction = home_team
    elif away_win > draw and away_win > home_win:
        prediction = away_team
    else:
        prediction = "Draw"

    confidence = max(home_win, draw, away_win)

    home_expected_points = (home_win * 3) + draw
    away_expected_points = (away_win * 3) + draw

    return {
        "Home Team": home_team,
        "Away Team": away_team,
        "Home Win %": round(home_win * 100, 1),
        "Draw %": round(draw * 100, 1),
        "Away Win %": round(away_win * 100, 1),
        "Predicted Result": prediction,
        "Home Expected Points": round(home_expected_points, 2),
        "Away Expected Points": round(away_expected_points, 2),
        "Confidence %": round(confidence * 100, 1)
    }

In [7]:
fixtures = [
    ("Arsenal", "Newcastle"),
    ("Arsenal", "Fulham"),
    ("West Ham", "Arsenal"),
    ("Arsenal", "Burnley"),
    ("Crystal Palace", "Arsenal"),

    ("Everton", "Man City"),
    ("Man City", "Brentford"),
    ("Bournemouth", "Man City"),
    ("Man City", "Crystal Palace"),
    ("Man City", "Aston Villa")
]

results = []

for home, away in fixtures:
    results.append(predict_match(home, away))

results_df = pd.DataFrame(results)
results_df

,Home Team,Away Team,Home Win %,Draw %,Away Win %,Predicted Result,Home Expected Points,Away Expected Points,Confidence %
0,Arsenal,Newcastle,70.9,19.5,9.5,Arsenal,2.32,0.48,70.9
1,Arsenal,Fulham,78.7,14.8,6.4,Arsenal,2.51,0.34,78.7
2,West Ham,Arsenal,11.9,18.0,70.0,Arsenal,0.54,2.28,70.0
3,Arsenal,Burnley,89.1,6.7,2.8,Arsenal,2.74,0.15,89.1
4,Crystal Palace,Arsenal,14.2,26.9,58.9,Arsenal,0.70,2.04,58.9
5,Everton,Man City,19.5,24.5,56.0,Man City,0.83,1.92,56.0
6,Man City,Brentford,74.8,15.7,9.4,Man City,2.40,0.44,74.8
7,Bournemouth,Man City,24.8,26.1,49.1,Man City,1.01,1.73,49.1
8,Man City,Crystal Palace,61.7,22.7,15.6,Man City,2.08,0.70,61.7
9,Man City,Aston Villa,71.5,17.5,10.9,Man City,2.32,0.50,71.5


In [8]:
arsenal_points = 0
city_points = 0

for index, row in results_df.iterrows():
    if row["Home Team"] == "Arsenal":
        arsenal_points += row["Home Expected Points"]
    elif row["Away Team"] == "Arsenal":
        arsenal_points += row["Away Expected Points"]

    if row["Home Team"] == "Man City":
        city_points += row["Home Expected Points"]
    elif row["Away Team"] == "Man City":
        city_points += row["Away Expected Points"]

arsenal_points, city_points

(11.89, 10.450000000000001)

In [9]:
arsenal_current_points = 70
city_current_points = 70

arsenal_final = arsenal_current_points + arsenal_points
city_final = city_current_points + city_points

arsenal_final, city_final

(81.89, 80.45)

In [10]:
num_simulations = 10000

arsenal_titles = 0
city_titles = 0
ties = 0

for sim in range(num_simulations):
    arsenal_total = arsenal_current_points
    city_total = city_current_points

    for index, row in results_df.iterrows():

        home_team = row["Home Team"]
        away_team = row["Away Team"]

        home_win_prob = row["Home Win %"] / 100
        draw_prob = row["Draw %"] / 100
        away_win_prob = row["Away Win %"] / 100

        total = home_win_prob + draw_prob + away_win_prob
        home_win_prob /= total
        draw_prob /= total
        away_win_prob /= total

        outcome = np.random.choice(
            ["Home Win", "Draw", "Away Win"],
            p=[home_win_prob, draw_prob, away_win_prob]
        )

        if outcome == "Home Win":
            if home_team == "Arsenal":
                arsenal_total += 3
            elif home_team == "Man City":
                city_total += 3

        elif outcome == "Draw":
            if home_team == "Arsenal" or away_team == "Arsenal":
                arsenal_total += 1
            if home_team == "Man City" or away_team == "Man City":
                city_total += 1

        elif outcome == "Away Win":
            if away_team == "Arsenal":
                arsenal_total += 3
            elif away_team == "Man City":
                city_total += 3

    if arsenal_total > city_total:
        arsenal_titles += 1
    elif city_total > arsenal_total:
        city_titles += 1
    else:
        ties += 1

arsenal_title_prob = arsenal_titles / num_simulations * 100
city_title_prob = city_titles / num_simulations * 100
tie_prob = ties / num_simulations * 100

arsenal_title_prob, city_title_prob, tie_prob

(58.74, 28.29, 12.97)

## Conclusion

The Poisson model predicts that both Arsenal and Manchester City are most likely to win all five of their remaining matches. However, Arsenal has the higher expected points total.

Arsenal is projected to earn 11.89 expected points from the remaining fixtures, while Manchester City is projected to earn 10.45 expected points.

After adding the current points, the model predicts:

- Arsenal predicted final points: 81.89
- Manchester City predicted final points: 80.45

The Monte Carlo simulation also supports Arsenal as the more likely title winner:

- Arsenal title probability: 59.0%
- Manchester City title probability: 28.7%
- Tie probability: 12.3%

Based on this model, Arsenal is projected to finish ahead of Manchester City. However, this model does not account for injuries, tactical changes, red cards, squad rotation, or other unpredictable match events.